# Notebook 3 — Training, Validation & Testing (Model Comparison + Ensemble)
### Immersion Aluminium Holding Furnace (1120 kg/ch) — Anomaly Detection Pipeline

**Purpose:** Train and compare three anomaly-detection approaches — **One-Class SVM**,
**Local Outlier Factor**, and a **supervised XGBoost classifier** trained on the Layer 1
rule-based labels — then combine them with the rule engine into a final **2-of-3 ensemble**.


In [ ]:
# =========================================================
# 0. ENVIRONMENT SETUP
# =========================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, confusion_matrix, classification_report, roc_curve)

try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except ImportError:
    from lightgbm import LGBMClassifier
    XGB_AVAILABLE = False

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
pd.set_option("display.max_columns", None)

INPUT_PATH = "/kaggle/input/notebooks/aliciakyoumi/m15-cia-02-processing"
OUTPUT_PATH = "/kaggle/working/"
RANDOM_STATE = 42

df = pd.read_csv(os.path.join(INPUT_PATH, "labeled_data.csv"), parse_dates=["timestamp"])
print("Loaded labeled data:", df.shape)
print("Anomaly rows in FULL labeled data:", int(df['is_anomaly'].sum()),
      f"({df['is_anomaly'].mean()*100:.4f}% of rows)")
df.head()


## 1. Train / Validation / Test Split

**Diagnostic note:** if the anomaly count printed above is small (a handful of rows out
of a large time series), a naive chronological 70/15/15 cut can accidentally place
*all* anomalies inside the train partition, leaving validation/test with **zero**
positive samples — metrics would then look empty even though the rule engine is
working correctly. To avoid this we:

1. **Impute instead of drop** rows with `NaN` resistance (caused by `current_avr == 0`,
   which is exactly the kind of partial-zero row we deliberately kept in Notebook 1 —
   dropping it here would silently delete real anomaly rows before the split ever happens).
2. Use a **stratified-in-time split**: anomaly rows and normal rows are each split
   70/15/15 along their own timeline, then recombined and re-sorted by timestamp.
   This guarantees every partition contains a proportional share of anomalies while
   still respecting chronological order *within* each class.


In [ ]:
# =========================================================
# 1a. FEATURE SELECTION + NaN HANDLING (impute, don't drop)
# =========================================================
FEATURE_COLS = [
    "molten_temp", "heater1", "heater2", "voltage_avr", "current_avr", "power_total",
    "resistance", "delta_temp_1h", "heater_active", "heater_both_off",
    "current_avr_roll_std", "power_total_roll_std"
]
TARGET_COL_BIN = "is_anomaly"       # binary anomaly flag -> used by OCSVM / LOF / ensemble
TARGET_COL_MULTI = "severity_rank"  # 4-class severity (0=NORMAL ... 3=FATAL/EMERGENCY) -> used by XGBoost/LightGBM

df_model = df.copy()

# resistance is NaN only when current_avr == 0 (division by zero) -- this is itself
# informative (e.g. relates to Uncontrolled Heating / short-circuit rows), so we impute
# with a sentinel far outside the normal operating range rather than deleting the row.
resistance_sentinel = df_model["resistance"].max() * 2 if df_model["resistance"].notna().any() else 0
df_model["resistance"] = df_model["resistance"].fillna(resistance_sentinel)

# Any remaining NaNs in other engineered features (rare -- only possible at series edges)
n_before = len(df_model)
df_model = df_model.dropna(subset=FEATURE_COLS + [TARGET_COL_BIN, TARGET_COL_MULTI]).reset_index(drop=True)
print(f"Rows dropped due to remaining NaNs (non-resistance): {n_before - len(df_model)}")
print("Rows available for modeling:", len(df_model))
print("Anomaly rows retained for modeling:", int(df_model[TARGET_COL_BIN].sum()))
print("\nSeverity class distribution (severity_rank, 0=NORMAL ... 3=FATAL/EMERGENCY):")
print(df_model[TARGET_COL_MULTI].value_counts().sort_index())


In [ ]:
# =========================================================
# 1b. STRATIFIED-IN-TIME 70/15/15 SPLIT
# Split each severity_rank class separately along the timeline, then recombine.
# This preserves chronological order within each class while guaranteeing every
# partition (train/val/test) contains all 4 severity classes proportionally.
# =========================================================
def stratified_time_split(df_in, target_col, train_frac=0.70, val_frac=0.15):
    parts_train, parts_val, parts_test = [], [], []
    for cls_value, group in df_in.groupby(target_col):
        group = group.sort_values("timestamp")
        n = len(group)
        t_end = int(n * train_frac)
        v_end = int(n * (train_frac + val_frac))
        parts_train.append(group.iloc[:t_end])
        parts_val.append(group.iloc[t_end:v_end])
        parts_test.append(group.iloc[v_end:])
    train_df = pd.concat(parts_train).sort_values("timestamp").reset_index(drop=True)
    val_df   = pd.concat(parts_val).sort_values("timestamp").reset_index(drop=True)
    test_df  = pd.concat(parts_test).sort_values("timestamp").reset_index(drop=True)
    return train_df, val_df, test_df

# Stratify on the 4-class severity_rank (not the binary flag) so WARNING / CRITICAL /
# FATAL-EMERGENCY rows are all proportionally represented in train, val, and test.
train_df, val_df, test_df = stratified_time_split(df_model, TARGET_COL_MULTI)

X_train = train_df[FEATURE_COLS]
X_val   = val_df[FEATURE_COLS]
X_test  = test_df[FEATURE_COLS]

# Binary targets (OCSVM / LOF / ensemble / evaluation ground truth)
y_train_bin, y_val_bin, y_test_bin = train_df[TARGET_COL_BIN], val_df[TARGET_COL_BIN], test_df[TARGET_COL_BIN]
# Multi-class targets (XGBoost / LightGBM severity classifier)
y_train_multi, y_val_multi, y_test_multi = train_df[TARGET_COL_MULTI], val_df[TARGET_COL_MULTI], test_df[TARGET_COL_MULTI]

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print(f"Anomaly count (bin)  -> train: {y_train_bin.sum()}, val: {y_val_bin.sum()}, test: {y_test_bin.sum()}")
print(f"Anomaly rate  (bin)  -> train: {y_train_bin.mean():.4f}, val: {y_val_bin.mean():.4f}, test: {y_test_bin.mean():.4f}")
print("\nSeverity class counts (multi) -> train:")
print(y_train_multi.value_counts().sort_index())
print("Severity class counts (multi) -> val:")
print(y_val_multi.value_counts().sort_index())
print("Severity class counts (multi) -> test:")
print(y_test_multi.value_counts().sort_index())

if y_test_bin.sum() == 0:
    print("\nWARNING: test set still has 0 anomalies -- your total anomaly count is likely "
          "too small (<~7 rows) for a 15% split to guarantee coverage. Consider lowering "
          "train_frac/val_frac, using k-fold time-series CV instead of a single split, or "
          "revisiting the Layer-1 thresholds in Notebook 2 if the rules are firing too rarely.")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)


## 2. Model 1 — One-Class SVM (Unsupervised)
Trained **only on rows labeled normal** in the training set, to learn the boundary
of "normal" furnace operating behavior.

In [ ]:
# =========================================================
# 2. ONE-CLASS SVM
# =========================================================
# "Normal" = severity_rank 0 (multi-class) OR is_anomaly 0 (binary) -- either
# labeling scheme agrees a row is normal, so we accept it if either says so.
normal_mask = (y_train_multi.values == 0) | (y_train_bin.values == 0)
X_train_normal = X_train_scaled[normal_mask]
print("Training OCSVM on normal-only samples:", X_train_normal.shape)

ocsvm = OneClassSVM(kernel="rbf", nu=0.05, gamma="scale")
ocsvm.fit(X_train_normal)

# OCSVM outputs +1 (normal) / -1 (outlier) -> convert to 0/1 anomaly convention
def ocsvm_predict(model, X_scaled):
    raw = model.predict(X_scaled)
    return np.where(raw == -1, 1, 0)

val_pred_ocsvm  = ocsvm_predict(ocsvm, X_val_scaled)
test_pred_ocsvm = ocsvm_predict(ocsvm, X_test_scaled)
# decision_function: higher = more normal; flip sign so higher = more anomalous (for ROC-AUC)
test_score_ocsvm = -ocsvm.decision_function(X_test_scaled)

print("OCSVM validation anomaly rate:", val_pred_ocsvm.mean().round(3))


In [ ]:
# =========================================================
# OCSVM Parameter Sensitivity Curve (Sebagai pengganti loss curve)
# =========================================================
nu_range = [0.01, 0.03, 0.05, 0.10, 0.15, 0.20]
val_f1_scores = []

for nu_val in nu_range:
    temp_ocsvm = OneClassSVM(kernel="rbf", nu=nu_val, gamma="scale")
    temp_ocsvm.fit(X_train_normal)
    temp_preds = np.where(temp_ocsvm.predict(X_val_scaled) == -1, 1, 0)
    val_f1_scores.append(f1_score(y_val_bin, temp_preds, zero_division=0))

# Plotting menggunakan Plotly
fig = px.line(x=nu_range, y=val_f1_scores, markers=True,
              labels={"x": "Nu Parameter (Nu)", "y": "Validation F1-Score"},
              title="One-Class SVM — Validation Curve (Nu vs F1-Score)")
fig.show()


## 3. Model 2 — Local Outlier Factor (Unsupervised, Density-Based)

In [ ]:
# =========================================================
# 3. LOCAL OUTLIER FACTOR
# LOF (novelty=True) supports predicting on unseen data after fitting on training data.
# =========================================================
lof = LocalOutlierFactor(n_neighbors=35, contamination=0.05, novelty=True)
lof.fit(X_train_scaled)

def lof_predict(model, X_scaled):
    raw = model.predict(X_scaled)
    return np.where(raw == -1, 1, 0)

val_pred_lof  = lof_predict(lof, X_val_scaled)
test_pred_lof = lof_predict(lof, X_test_scaled)
test_score_lof = -lof.decision_function(X_test_scaled)

print("LOF validation anomaly rate:", val_pred_lof.mean().round(3))


In [ ]:
# =========================================================
# Distribusi Skor Anomali (OCSVM & LOF pada Test Set)
# =========================================================
fig = make_subplots(rows=1, cols=2, subplot_titles=("OCSVM Score Distribution", "LOF Score Distribution"))

# Plot OCSVM Scores
fig.add_trace(go.Histogram(x=test_score_ocsvm[y_test_bin == 0], name="Normal (OCSVM)", marker_color="steelblue", opacity=0.7), row=1, col=1)
fig.add_trace(go.Histogram(x=test_score_ocsvm[y_test_bin == 1], name="Anomaly (OCSVM)", marker_color="firebrick", opacity=0.9), row=1, col=1)

# Plot LOF Scores
fig.add_trace(go.Histogram(x=test_score_lof[y_test_bin == 0], name="Normal (LOF)", marker_color="steelblue", opacity=0.7, showlegend=False), row=1, col=2)
fig.add_trace(go.Histogram(x=test_score_lof[y_test_bin == 1], name="Anomaly (LOF)", marker_color="firebrick", opacity=0.9, showlegend=False), row=1, col=2)

fig.update_layout(barmode="overlay", title="Distribusi Skor Anomali Model Unsupervised pada Test Set", height=400)
fig.show()


## 4. Model 3 — XGBoost Classifier (Supervised on Layer 1 Labels)

In [ ]:
# =========================================================
# 4. XGBOOST / LIGHTGBM CLASSIFIER (MULTI-CLASS SEVERITY, 4 KELAS)
# =========================================================
# Class weighting: compute_sample_weight('balanced', ...) reweights each row by the
# inverse frequency of its severity_rank class, so rare CRITICAL / FATAL-EMERGENCY
# rows aren't drowned out by the NORMAL majority -- this replaces scale_pos_weight,
# which only works for binary targets.
sample_weights = compute_sample_weight(class_weight="balanced", y=y_train_multi)

if XGB_AVAILABLE:
    clf = XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        objective="multi:softprob", num_class=4,
        eval_metric="mlogloss", random_state=RANDOM_STATE,
    )
    # Track BOTH train and val mlogloss per boosting round (this is XGBoost's
    # equivalent of a "training curve" -- 300 rounds instead of 300 epochs).
    clf.fit(X_train, y_train_multi, sample_weight=sample_weights,
            eval_set=[(X_train, y_train_multi), (X_val, y_val_multi)], verbose=False)
    eval_history = clf.evals_result()
    train_logloss = eval_history["validation_0"]["mlogloss"]
    val_logloss   = eval_history["validation_1"]["mlogloss"]
else:
    clf = LGBMClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE,
        objective="multiclass", num_class=4,
    )
    clf.fit(X_train, y_train_multi, sample_weight=sample_weights,
            eval_set=[(X_train, y_train_multi), (X_val, y_val_multi)],
            eval_names=["train", "val"], eval_metric="multi_logloss")
    eval_history = clf.evals_result_
    train_logloss = eval_history["train"]["multi_logloss"]
    val_logloss   = eval_history["val"]["multi_logloss"]

val_pred_clf_multi   = clf.predict(X_val)          # 0-3 severity class
test_pred_clf        = clf.predict(X_test)         # 0-3 severity class (raw multi-class prediction)
test_score_clf_multi = clf.predict_proba(X_test)   # shape (n_test, 4) -- P(class 0..3)

print("Classifier validation accuracy (multi-class):", (val_pred_clf_multi == y_val_multi.values).mean().round(3))
print(f"Boosting rounds completed: {len(train_logloss)}")
print(f"Final train mlogloss: {train_logloss[-1]:.4f} | Final val mlogloss: {val_logloss[-1]:.4f}")


### Why this "trains" fast despite having 300 boosting rounds

Each boosting round fits a shallow decision tree (`max_depth=5`) to the current
residual error — this is a closed-form greedy split search over the feature set,
not a gradient-descent weight update like in a neural network. With ~12 features and
a tabular dataset of this size, XGBoost's C++ engine can evaluate all 300 rounds in
well under a second on CPU. The train/val logloss curve below is the direct visual
proof that iterative learning *is* happening — just much faster than deep learning epochs.

In [ ]:
# XGBoost Training Curve (Multi-class Log Loss)
fig = make_subplots(rows=1, cols=2, subplot_titles=("Train mlogloss", "Validation mlogloss"))
fig.add_trace(go.Scatter(y=train_logloss, mode="lines", name="Train", line=dict(color="steelblue")), row=1, col=1)
fig.add_trace(go.Scatter(y=val_logloss, mode="lines", name="Validation", line=dict(color="firebrick")), row=1, col=2)
fig.update_layout(title="XGBoost Training Loss Curve (mlogloss / multi_logloss)", height=400, showlegend=False)
fig.show()


In [ ]:
# =========================================================
# 4a. Feature importance
# =========================================================
importances = pd.Series(clf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
fig = px.bar(importances.reset_index(), x=0, y="index", orientation="h",
             labels={"0": "Importance", "index": "Feature"},
             title="XGBoost Feature Importance")
fig.update_yaxes(categoryorder="total ascending")
fig.show()


## 5. Evaluation — Accuracy, Precision, Recall, F1, ROC-AUC, Confusion Matrices

All three models are evaluated on the held-out **test set**, against the Layer 1 rule labels
(used here as ground truth for supervised-style comparison).

In [ ]:
# =========================================================
# 5a0. Binarize XGBoost/LightGBM multi-class output for anomaly-detection evaluation
# The classifier now predicts 4 severity classes (0=NORMAL..3=FATAL/EMERGENCY), but
# evaluate() / the ensemble / the confusion matrices & ROC curves below all compare
# against the binary is_anomaly ground truth -- so we collapse classes 1/2/3 -> anomaly.
# =========================================================
test_pred_clf_bin  = np.where(test_pred_clf > 0, 1, 0)            # class 1, 2 or 3 -> anomaly
test_score_clf_bin = 1.0 - test_score_clf_multi[:, 0]              # 1 - P(class 0 / Normal)

# =========================================================
# 5a. Metrics table
# =========================================================
def evaluate(y_true, y_pred, y_score=None, name=""):
    row = {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
    }
    if y_score is not None:
        try:
            row["roc_auc"] = roc_auc_score(y_true, y_score)
        except ValueError:
            row["roc_auc"] = np.nan
    else:
        row["roc_auc"] = np.nan
    return row

results = pd.DataFrame([
    evaluate(y_test_bin, test_pred_ocsvm, test_score_ocsvm, "One-Class SVM"),
    evaluate(y_test_bin, test_pred_lof, test_score_lof, "Local Outlier Factor"),
    evaluate(y_test_bin, test_pred_clf_bin, test_score_clf_bin, "XGBoost (Supervised, binarized)"),
])
results.round(4)


In [ ]:
# =========================================================
# 5b. Metrics comparison bar chart
# =========================================================
metrics_long = results.melt(id_vars="model", value_vars=["accuracy", "precision", "recall", "f1_score", "roc_auc"],
                             var_name="metric", value_name="score")
fig = px.bar(metrics_long, x="metric", y="score", color="model", barmode="group",
             title="Model Comparison — Test Set Metrics")
fig.show()


In [ ]:
# =========================================================
# 5c. Confusion matrices (side-by-side)
# =========================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (preds, name) in zip(axes, [(test_pred_ocsvm, "One-Class SVM"),
                                      (test_pred_lof, "Local Outlier Factor"),
                                      (test_pred_clf_bin, "XGBoost")]):
    cm = confusion_matrix(y_test_bin, preds)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Normal", "Anomaly"], yticklabels=["Normal", "Anomaly"])
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.suptitle("Confusion Matrices — Test Set")
plt.tight_layout()
plt.show()


In [ ]:
# =========================================================
# 5d. ROC curves
# =========================================================
fig = go.Figure()
for score, name in [(test_score_ocsvm, "One-Class SVM"), (test_score_lof, "Local Outlier Factor"),
                     (test_score_clf_bin, "XGBoost")]:
    fpr, tpr, _ = roc_curve(y_test_bin, score)
    auc_val = roc_auc_score(y_test_bin, score)
    fig.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines", name=f"{name} (AUC={auc_val:.3f})"))

fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Random", line=dict(dash="dash", color="gray")))
fig.update_layout(title="ROC Curves — Model Comparison", xaxis_title="False Positive Rate",
                   yaxis_title="True Positive Rate", height=500)
fig.show()


## 6. Ensemble Logic (Layer 1 + Layer 2 Voting)

Final decision rule:

> **Flag anomaly if** the Layer-1 rule engine fires, **OR** at least **2 of the 3 ML
> models** (OCSVM, LOF, XGBoost) independently flag the sample as anomalous.


In [ ]:
# =========================================================
# 6. ENSEMBLE FUNCTION DENGAN SKOR PROBABILITAS KONTINU (UNTUK ROC-AUC)
# =========================================================

def min_max_scale(scores):
    s_min, s_max = np.min(scores), np.max(scores)
    if s_max - s_min == 0:
        return np.zeros_like(scores)
    return (scores - s_min) / (s_max - s_min)

prob_ocsvm = min_max_scale(test_score_ocsvm)
prob_lof   = min_max_scale(test_score_lof)
prob_xgb   = test_score_clf_bin  # already a 0-1 anomaly probability (1 - P(Normal))

test_score_ensemble_ml = (prob_ocsvm + prob_lof + prob_xgb) / 3.0

def ensemble_predict(rule_flag, ocsvm_flag, lof_flag, clf_flag):
    rule_flag  = np.asarray(rule_flag)
    ml_votes   = np.asarray(ocsvm_flag) + np.asarray(lof_flag) + np.asarray(clf_flag)
    ml_trigger = (ml_votes >= 2).astype(int)
    return np.maximum(rule_flag, ml_trigger)

# Layer-1 rule flag: Notebook 2 (current version) no longer stores separate boolean
# rule_* columns -- individual conditions were consolidated into "reason" /
# "severity_level" / "n_triggered_reasons". We reconstruct the same "did any Layer-1
# rule fire" signal from whichever of those consolidated columns is available,
# falling back gracefully for older/newer labeled_data.csv schemas.
rule_cols = ["rule_uncontrolled_heating", "rule_short_circuit", "rule_thermal_lag",
             "rule_overheat", "rule_low_temp"]

if all(col in test_df.columns for col in rule_cols):
    # Legacy schema: separate boolean rule_* columns
    test_rule_flag = test_df[rule_cols].max(axis=1).values
elif "n_triggered_reasons" in test_df.columns:
    # Current schema: count of Layer-1 conditions triggered per row
    test_rule_flag = (test_df["n_triggered_reasons"] > 0).astype(int).values
elif "reason" in test_df.columns:
    # Fallback: single consolidated root-cause label
    test_rule_flag = (test_df["reason"] != "Normal Operation").astype(int).values
else:
    # Last resort: is_anomaly was itself derived from the Layer-1 rule engine in Notebook 2
    test_rule_flag = y_test_bin.values

test_ensemble_pred = ensemble_predict(test_rule_flag, test_pred_ocsvm, test_pred_lof, test_pred_clf_bin)

ensemble_result = evaluate(
    y_test_bin,
    test_ensemble_pred,
    y_score=test_score_ensemble_ml,
    name="Final Ensemble (Rule OR 2-of-3 ML)"
)

results_with_ensemble = pd.concat([results, pd.DataFrame([ensemble_result])], ignore_index=True)

# =========================================================
# KONVERSI KE FORMAT PERSENTASE (%)
# =========================================================
# Membuat salinan dataframe khusus untuk ditampilkan dalam bentuk persentase
results_percentage = results_with_ensemble.copy()
numeric_cols = ["accuracy", "precision", "recall", "f1_score", "roc_auc"]

for col in numeric_cols:
    # Dikalikan 100 dan dibulatkan 2 desimal, lalu diubah ke string berformat "%"
    results_percentage[col] = results_percentage[col].apply(
        lambda x: f"{x * 100:.2f}%" if pd.notnull(x) else "NaN"
    )


In [ ]:
# =========================================================
# 6a. Ensemble confusion matrix
# =========================================================
plt.figure(figsize=(5, 4))
cm = confusion_matrix(y_test_bin, test_ensemble_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples",
            xticklabels=["Normal", "Anomaly"], yticklabels=["Normal", "Anomaly"])
plt.title("Final Ensemble — Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()


## 7. Save Models

In [ ]:
# =========================================================
# 7. SAVE ARTIFACTS
# =========================================================
joblib.dump(ocsvm, os.path.join(OUTPUT_PATH, "model_ocsvm.pkl"))
joblib.dump(lof,   os.path.join(OUTPUT_PATH, "model_lof.pkl"))
joblib.dump(clf,   os.path.join(OUTPUT_PATH, "model_xgb.pkl"))
joblib.dump(scaler, os.path.join(OUTPUT_PATH, "scaler.pkl"))

# Save the exact feature order + rule thresholds so the dashboard can reconstruct engineered features
import json
metadata = {
    "feature_cols": FEATURE_COLS,
    "core_features": ["molten_temp", "heater1", "heater2", "voltage_avr", "current_avr", "power_total"],
    "overheat_threshold": 680,
    "low_temp_threshold": 640,
    "thermal_lag_delta": -5,
}
with open(os.path.join(OUTPUT_PATH, "model_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved: model_ocsvm.pkl, model_lof.pkl, model_xgb.pkl, scaler.pkl, model_metadata.json")
results_percentage
